# Day 1 Instructor Demo. What is an LLM, and which kind do I want?

**Instructor notebook.** Runs on paid Colab Pro with GPU.

Three sections, matching the slide deck.

- **Section A**. Three-line encoder demo on five political sentences. Cardiff Twitter-RoBERTa sentiment. Mirrors slide 20.
- **Section B**. CPU vs GPU timing on 1000 UN-speech sentences with the same model. Mirrors slide 21.
- **Section C**. Zero-shot stance on roughly 200 UN General Debate Corpus 2017 sentences with DeBERTa-v3 NLI. Mirrors slides 26 to 27.

Pre-flight before going live. Run cells 1 through 4 once on the connected runtime so model weights are cached. Disconnect and reconnect once to confirm warm starts.

## 0. Setup

One install. One device check. The rest is just `transformers`.

In [ ]:
!pip install -q transformers==4.44.2

In [ ]:
import torch
import platform

print('Python', platform.python_version())
print('Torch', torch.__version__)
print('CUDA available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device', torch.cuda.get_device_name(0))

---
## Section A. Three lines. Five political sentences.

The model is `cardiffnlp/twitter-roberta-base-sentiment-latest`. About 125 million parameters. CC-BY-4.0. It returns one of three labels (negative, neutral, positive) with a confidence score.

Five sentences below cover three substantive contexts. Read each output dictionary aloud during the demo. Highlight the `label` and `score` keys.

In [ ]:
from transformers import pipeline

clf = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    device=0 if torch.cuda.is_available() else -1,
)
print('Cardiff sentiment pipeline ready.')

In [ ]:
five_sentences = [
    'The new healthcare bill is a disaster for working families.',
    "Today's bipartisan infrastructure vote was a real step forward.",
    'The government has yet to clarify its position on the upcoming summit.',
    'Years of austerity have undermined public trust in the institutions of the European Union.',
    'We welcome the historic ceasefire announcement and look forward to a durable peace.',
]

for i, s in enumerate(five_sentences, 1):
    out = clf(s)[0]
    print(f"[{i}] {out['label']:<8}  ({out['score']:.2f})  {s}")

**What to point out.** Three labels, three probabilities. The model has never seen these sentences before. The output is a Python dict, the same structure every call. That regular structure is what makes encoders easy to scale.

---
## Section B. CPU vs GPU on 1000 UN-speech sentences.

We download the UN General Debate Corpus, sentence-tokenize 2017 speeches, take 1000 sentences, and run the same Cardiff sentiment model twice. Once on CPU, once on GPU.

**Live procedure.** Run the load cell. Run the CPU benchmark cell. Read the wall time aloud. Switch runtime to T4 GPU (Runtime, Change runtime type, T4 GPU, Save). Re-run the load cell to repopulate state on the new runtime, then re-run the GPU benchmark cell. Read the wall time. Plot.

In [ ]:
import os
import re
import urllib.request
import pandas as pd

UNGDC_URL = (
    'https://dataverse.harvard.edu/api/access/datafile/'
    ':persistentId?persistentId=doi:10.7910/DVN/0TJX8Y/2CIIBR'
)
LOCAL_PATH = 'ungdc.csv'

if not os.path.exists(LOCAL_PATH):
    print('Downloading UNGDC...')
    urllib.request.urlretrieve(UNGDC_URL, LOCAL_PATH)

df = pd.read_csv(LOCAL_PATH)
print('UNGDC shape', df.shape)
print('Columns', df.columns.tolist())
print()
print('First 2017 row:')
row_2017 = df.loc[df['year'] == 2017].iloc[0]
print(' country', row_2017['country'])
print(' year', row_2017['year'])
print(' text head', repr(row_2017['text'][:200]))

In [ ]:
import random

SENT_RE = re.compile(r'(?<=[.!?])\s+(?=[A-Z])')

def split_sentences(text):
    out = [s.strip() for s in SENT_RE.split(text) if s.strip()]
    return [s for s in out if 40 <= len(s) <= 400]

speeches_2017 = df.loc[df['year'] == 2017, 'text'].tolist()
all_sents = []
for s in speeches_2017:
    all_sents.extend(split_sentences(s))

random.seed(42)
random.shuffle(all_sents)
thousand = all_sents[:1000]
two_hundred = all_sents[1000:1200]
print('1000-sentence batch ready for Section B.')
print('200-sentence stance sample reserved for Section C.')
print('Sample sentence:', thousand[0][:160])

In [ ]:
import time

clf_cpu = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment-latest',
    device=-1,
    truncation=True,
)

t0 = time.time()
out_cpu = clf_cpu(thousand, batch_size=16)
elapsed_cpu = time.time() - t0
print(f'CPU pass over 1000 sentences. {elapsed_cpu:.1f} seconds.')

In [ ]:
if not torch.cuda.is_available():
    print('No GPU attached. Switch runtime to T4 in Runtime, Change runtime type.')
    elapsed_gpu = float('nan')
else:
    clf_gpu = pipeline(
        'sentiment-analysis',
        model='cardiffnlp/twitter-roberta-base-sentiment-latest',
        device=0,
        truncation=True,
    )
    _ = clf_gpu(thousand[:32], batch_size=32)  # warm up
    t0 = time.time()
    out_gpu = clf_gpu(thousand, batch_size=64)
    elapsed_gpu = time.time() - t0
    print(f'GPU pass over 1000 sentences. {elapsed_gpu:.1f} seconds.')
    print(f'Speedup. {elapsed_cpu / elapsed_gpu:.1f}x.')

In [ ]:
import matplotlib.pyplot as plt

labels = ['Free CPU', 'T4 GPU']
values = [elapsed_cpu, elapsed_gpu]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(labels, values, color=['#5C7B6A', '#334B99'])
for i, v in enumerate(values):
    if v == v:  # not NaN
        ax.text(v + max(values) * 0.02, i, f'{v:.0f}s', va='center')
ax.set_xlabel('Wall time, seconds')
ax.set_title('1000-sentence Cardiff sentiment pass')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**What to point out.** The number on the right is what we paid for. The number on the left is what we will not run again. Once you understand the speedup, you stop avoiding tasks that take hours. You run them while you make coffee.

---
## Section C. Zero-shot stance on UN climate language.

Switch from a sentiment classifier to an NLI-trained encoder. Same architecture. Different head. Different question.

We feed `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` three plain-English candidate labels and let it score the 200-sentence sample we set aside in Section B.

In [ ]:
zsc = pipeline(
    'zero-shot-classification',
    model='MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli',
    device=0 if torch.cuda.is_available() else -1,
)

candidate_labels = [
    'the speaker supports stronger international climate action',
    'the speaker is skeptical of international climate action',
    'the speaker does not discuss climate',
]

print('DeBERTa zero-shot pipeline ready.')
print('Candidate labels:')
for lbl in candidate_labels:
    print(' -', lbl)

In [ ]:
t0 = time.time()
results = zsc(two_hundred, candidate_labels=candidate_labels, batch_size=16)
elapsed = time.time() - t0
print(f'200-sentence zero-shot pass. {elapsed:.1f} seconds.')

In [ ]:
scored = pd.DataFrame({
    'sentence': two_hundred,
    'top_label': [r['labels'][0] for r in results],
    'top_score': [r['scores'][0] for r in results],
    'second_label': [r['labels'][1] for r in results],
    'second_score': [r['scores'][1] for r in results],
})

print('Prediction distribution:')
print(scored['top_label'].value_counts())
print()
print('Confidence summary:')
print(scored['top_score'].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(scored['top_score'], bins=20, color='#5C7B6A', edgecolor='white')
ax.axvline(0.5, color='#334B99', linestyle='--', label='argmax floor (0.5)')
ax.set_xlabel('Top-label probability')
ax.set_ylabel('Sentence count')
ax.set_title('Calibration. Where does the model commit?')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### Three confident hits

Pick three rows where `top_score > 0.85` and the label is the supportive class. Read aloud. The lexical signal is doing the work.

In [ ]:
supportive = scored.loc[
    (scored['top_label'] == candidate_labels[0]) & (scored['top_score'] > 0.85)
].head(3)

for _, row in supportive.iterrows():
    print(f"[{row['top_score']:.2f}] {row['sentence'][:280]}")
    print()

### Five misclassifications and why

Read aloud the manual annotations beneath each row. These are the five-misses pedagogical moment from slide 27.

In [ ]:
look_at = scored.loc[
    (scored['top_score'] > 0.45) & (scored['top_score'] < 0.65)
].head(5)

for i, (_, row) in enumerate(look_at.iterrows(), 1):
    print(f'--- Miss {i} ---')
    print(f"top: {row['top_label']} ({row['top_score']:.2f})")
    print(f"runner-up: {row['second_label']} ({row['second_score']:.2f})")
    print(row['sentence'][:300])
    print()

**Speaker patter for each miss.** Match the live output to one of these patterns and read the matching diagnosis.

1. **Hidden lexicon.** A sentence about energy independence with no climate keyword lands off-topic. The model has no climate-relevant lexicon attached to "energy security" without explicit cues.
2. **Mixed paragraph.** A sentence that praises Paris implementation but opens with criticism of slow progress lands skeptical. The model picked the first sentiment cue and missed the second.
3. **Low-confidence argmax.** Aspirational language at probability 0.51. The model is not confident, but argmax is what we read. Look at probabilities, not labels.
4. **Implicit framing.** A small-island statement about "rising seas" that lands correctly supportive but at low confidence. Climate is implicit, not lexical.
5. **Rhetoric.** Sarcasm or hedging in formal UN register lands the wrong way. Encoders are weak here.

**Where Day 2 goes from here.** Hand-coded gold set. Validation. Confusion matrix. Cohen's kappa. The whole point of tomorrow is to give these miss rates a number you can put in an appendix.

In [ ]:
scored.to_csv('day1_demo_scored.csv', index=False)
print('Saved day1_demo_scored.csv with', len(scored), 'rows.')